# TP 3 — Matplotlib & histogrammes

**Objectifs**

- Maîtriser l'affichage d'images avec Matplotlib (figures, sous-graphes, titres).
- Calculer et tracer l'histogramme d'intensité d'une image en niveaux de gris et l'histogramme par canal d'une image RGB.
- Implémenter manuellement deux techniques d'amélioration de contraste :
  - étirement linéaire par percentiles ;
  - égalisation d'histogramme via la fonction de répartition cumulative.
- Comparer visuellement et statistiquement l'effet de ces traitements.

**Durée indicative : 50 minutes.**

In [1]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from skimage import data

# 'moon' est une image en niveaux de gris peu contrastée — idéale pour ce TP.
img_gray = np.array(Image.fromarray(data.moon()))  # (H, W), uint8
img_rgb = np.array(Image.fromarray(data.coffee()))  # (H, W, 3), uint8

print("gray :", img_gray.shape, img_gray.dtype, img_gray.min(), img_gray.max())
print("rgb  :", img_rgb.shape, img_rgb.dtype, img_rgb.min(), img_rgb.max())

gray : (512, 512) uint8 0 255
rgb  : (400, 600, 3) uint8 0 255


## Exercice 1 — Affichage propre et colorbar

Affichez `img_gray` et `img_rgb` côte à côte avec :

- une figure de taille raisonnable (`figsize=(10, 4)`),
- un titre pour chaque sous-graphe,
- les axes désactivés,
- `cmap="gray"` et `vmin=0, vmax=255` pour l'image en niveaux de gris.

**Ajout — colorbar avec une colormap perceptuelle** : créez ensuite **une troisième figure** affichant `img_gray` avec la colormap `"viridis"` et **une colorbar visible à droite** (`plt.colorbar(im, ax=ax)`). C'est l'affichage type quand on visualise une grandeur scalaire (carte de profondeur, heatmap d'activation, intensité d'un capteur).

<details>
<summary>💡 Coup de pouce</summary>

- Pour la colorbar, vous **devez** garder l'objet retourné par `imshow` : `im = ax.imshow(...)` puis `plt.colorbar(im, ax=ax)`.
- `fraction=0.046, pad=0.04` en argument de `plt.colorbar` rend la barre fine et bien alignée à droite de l'image.

</details>

In [2]:
# TODO : affichage côte à côte img_gray + img_rgb

# TODO : 2e figure avec colormap "viridis" + colorbar
# fig, ax = plt.subplots(figsize=(5, 4))
# im = ax.imshow(img_gray, cmap="viridis")
# plt.colorbar(im, ax=ax)

## Exercice 2 — Histogramme en niveaux de gris

1. Calculez l'histogramme de `img_gray` avec `np.histogram` (256 bins, plage `[0, 256]`).
2. Tracez-le en bâtonnets (`plt.bar`) ou en ligne continue (`plt.plot`).
3. Constatez visuellement que l'histogramme est concentré dans une plage étroite : `img_gray` est peu contrastée.

<details>
<summary>💡 Coup de pouce</summary>

- `np.histogram(img, bins=256, range=(0, 256))` renvoie un **tuple** `(hist, edges)` où `edges` a 257 valeurs (les bornes des bins).
- Pour `plt.bar`, utilisez les centres : `centers = (edges[:-1] + edges[1:]) / 2`.
- Pour `plt.plot`, vous pouvez tracer directement `hist` (l'indice = la valeur d'intensité).

</details>

In [3]:
# TODO

## Exercice 3 — Histogramme par canal RGB

Sur une même figure, tracez les histogrammes des canaux rouge, vert et bleu de `img_rgb`, en utilisant la couleur correspondante pour chaque courbe.

<details>
<summary>💡 Coup de pouce</summary>

- Itérez sur les 3 canaux : `for i, color in enumerate(["red", "green", "blue"]):`
- Récupérez le canal avec `img_rgb[..., i]` et tracez l'histogramme avec la couleur correspondante (`color=color` dans `plt.plot`).
- Ajoutez `plt.legend()` pour afficher les noms des canaux.

</details>

In [4]:
# TODO

## Exercice 4 — Étirement de contraste par percentiles

Implémentez la fonction suivante :

```python
def stretch(img, p_low=2, p_high=98):
    """Étire le contraste d'une image uint8 sur la plage des percentiles
    [p_low, p_high] vers [0, 255], avec clipping."""
```

Appliquez-la à `img_gray` et tracez :

- l'image d'origine et l'image étirée côte à côte,
- leurs histogrammes respectifs en-dessous (figure 2×2).

<details>
<summary>💡 Coup de pouce</summary>

- Récupérez les percentiles avec `a, b = np.percentile(img, [p_low, p_high])`.
- Formule : `out = (img - a) * 255 / (b - a)`. Passez d'abord en float pour éviter le débordement uint8.
- Terminez par `np.clip(out, 0, 255).astype(np.uint8)`.

</details>

In [5]:
# TODO

## Exercice 5 — Égalisation d'histogramme manuelle

Implémentez l'égalisation d'histogramme via la fonction de répartition cumulative :

```python
def equalize(img):
    """Égalise l'histogramme d'une image uint8 en niveaux de gris."""
```

Étapes :

1. Calculer l'histogramme `hist` sur 256 bins.
2. Calculer sa CDF normalisée à `[0, 255]`.
3. Remapper l'image par indexation : `out = cdf[img]`.
4. Cast en `uint8`.

Comparez les trois résultats — originale, étirée, égalisée — sur une figure 2×3 (images en haut, histogrammes en bas).

**Bonus** : ajoutez une quatrième version « correction gamma » avec γ = 0.6 :  
`gamma(img, γ) = clip( 255 * (img / 255) ** γ , 0, 255 )`. Lequel des trois traitements préférez-vous, et pourquoi ?

<details>
<summary>💡 Coup de pouce</summary>

- CDF : `cdf = hist.cumsum()` puis normalisation `cdf = 255 * cdf / cdf[-1]`.
- **Astuce** : `cdf[img]` fonctionne directement comme une LUT — NumPy indexe par tableau et renvoie une image transformée de même shape que `img`.
- Pour le **bonus gamma** : `out = 255 * (img / 255) ** g`. Penser au `clip` et au cast uint8.

</details>

In [6]:
# TODO